<a href="https://colab.research.google.com/github/KanjengRadenMasUnan/UTS-BIG-DATA/blob/main/uts-big-data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import matplotlib.pyplot as plt
import seaborn as sns

# Pastikan data NLTK yang diperlukan tersedia
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError: # Changed from nltk.downloader.DownloadError to LookupError
    nltk.download('vader_lexicon')

result, _ = reviews(
    'com.kemendikdasmen.rumahpendidikan',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

# --- Bagian baru untuk analisis sentimen dan grafik ---

# Memuat data dari CSV ke DataFrame
df = pd.read_csv(filename)

# Inisialisasi Sentiment Intensity Analyzer
sia = SentimentIntensityAnalyzer()

# Fungsi untuk mengklasifikasikan sentimen
def classify_sentiment(text):
    if isinstance(text, str):
        score = sia.polarity_scores(text)
        if score['compound'] >= 0.05:
            return 'Positive'
        elif score['compound'] <= -0.05:
            return 'Negative'
        else:
            return 'Neutral'
    return 'Neutral' # Handle non-string content or missing values

# Menerapkan analisis sentimen ke kolom 'content'
df['sentiment'] = df['content'].apply(classify_sentiment)

# Menghitung distribusi sentimen
sentiment_counts = df['sentiment'].value_counts()

# Membuat subplots untuk diagram batang dan diagram lingkaran
fig, axes = plt.subplots(1, 2, figsize=(16, 7)) # 1 baris, 2 kolom untuk 2 grafik

# Diagram Batang
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, hue=sentiment_counts.index, palette='viridis', legend=False, ax=axes[0])
axes[0].set_title('Distribusi Sentimen Ulasan Google Play (Bar Chart)')
axes[0].set_xlabel('Sentimen')
axes[0].set_ylabel('Jumlah Ulasan')

# Diagram Lingkaran
axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%', startangle=90, colors=sns.color_palette('viridis'))
axes[1].set_title('Distribusi Sentimen Ulasan Google Play (Pie Chart)')
axes[1].axis('equal') # Memastikan diagram lingkaran berbentuk lingkaran sempurna

plt.tight_layout() # Menyesuaikan tata letak agar tidak ada tumpang tindih

# Menyimpan hasil grafik ke dalam file PNG SEBELUM plt.show()
plt.savefig('distribusi_sentimen_ulasan_google_play.png')
print("Grafik distribusi sentimen telah berhasil disimpan sebagai 'distribusi_sentimen_ulasan_google_play.png'.")

plt.show() # Tampilkan plot setelah disimpan



print("Analisis sentimen dan grafik telah berhasil dibuat.")